## UCI Heart Disease Dataset

Date: 06-06-2026

Student ID: Mormoo5363

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from sklearn.decomposition import PCA

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

from IPython.display import display, Markdown

display(Markdown("## Load Dataset"))

# Load Heart Disease dataset
heart = pd.read_csv("heart_disease_uci.csv")

print("Dataset Shape:", heart.shape)

heart.head()

## Load Dataset

Dataset Shape: (920, 16)


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [4]:
display(Markdown("""
# Classification Task: Optimized SVM

### Goal
Predict whether a patient has heart disease.

### Optimization Technique
GridSearchCV will be used to identify the best values for C and gamma.

### Regularization
The C parameter serves as a regularization technique that helps reduce overfitting.

### Evaluation Metrics
- Accuracy
- Precision
- Recall
- F1 Score
"""))


# Classification Task: Optimized SVM

### Goal
Predict whether a patient has heart disease.

### Optimization Technique
GridSearchCV will be used to identify the best values for C and gamma.

### Regularization
The C parameter serves as a regularization technique that helps reduce overfitting.

### Evaluation Metrics
- Accuracy
- Precision
- Recall
- F1 Score


In [9]:
# Classification

display(Markdown("## Classification Task: Optimized SVM"))
display(Markdown("### Tuning with GridSearchCV and regularization using the C parameter."))

# Create binary target from UCI "num" column
# 0 = no heart disease, 1 = heart disease present
heart["heart_disease"] = heart["num"].apply(lambda x: 1 if x > 0 else 0)

# Drop columns that are not features
X = heart.drop(["id", "num", "heart_disease"], axis=1)
y = heart["heart_disease"]

# Fill missing values
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object", "bool"]).columns

for col in numeric_cols:
    X[col] = X[col].fillna(X[col].median())

for col in categorical_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

# One-hot encode categorical columns
X = pd.get_dummies(X, drop_first=True)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

# Scale data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Grid search parameters
param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", 0.01, 0.001],
    "kernel": ["rbf"]
}

grid_search = GridSearchCV(
    SVC(),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid_search.fit(X_train_scaled, y_train)

best_svm = grid_search.best_estimator_

y_pred = best_svm.predict(X_test_scaled)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

## Classification Task: Optimized SVM

### Tuning with GridSearchCV and regularization using the C parameter.

C:\Users\Morgan Moore\AppData\Local\Temp\ipykernel_16220\3804036268.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object", "bool"]).columns


Best Parameters:
{'C': 10, 'gamma': 0.001, 'kernel': 'rbf'}

Accuracy: 0.8260869565217391
Precision: 0.86
Recall: 0.8269230769230769
F1: 0.8431372549019608


## Classification Results

The SVM model was optimized using GridSearchCV to identify the best values for the C and gamma hyperparameters. The C parameter also serves as a regularization technique by controlling the tradeoff between maximizing the margin and minimizing classification errors.

### Findings

The optimized model was evaluated using accuracy, precision, recall, and F1-score. Compared to the original model from Part 3, the tuned model demonstrated improved generalization and reduced the likelihood of overfitting. Hyperparameter tuning helped identify the best parameter combination while maintaining strong predictive performance on unseen data.

In [10]:
try:
    from xgboost import XGBRegressor
    print("XGBoost Installed")
except:
    print("XGBoost NOT Installed")

XGBoost Installed


## Regression Task: Optimized XGBoost Regressor

### Goal
Predict patient cholesterol levels.

### Optimization Technique
GridSearchCV will be used to tune n_estimators, max_depth, and learning_rate.

### Regularization
L1 and L2 regularization will be applied using reg_alpha and reg_lambda to help reduce overfitting.

### Evaluation Metrics
- Mean Squared Error
- Root Mean Squared Error
- R-squared Score

In [11]:
from xgboost import XGBRegressor

display(Markdown("## Regression Task: Optimized XGBoost Regressor"))
display(Markdown("### Tuning with GridSearchCV and L1/L2 regularization."))

# Use rows where cholesterol is greater than 0
regression_data = heart[heart["chol"] > 0].copy()

X_reg = regression_data.drop(["id", "chol", "num", "heart_disease"], axis=1)
y_reg = regression_data["chol"]

# Fill missing values
numeric_cols = X_reg.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X_reg.select_dtypes(include=["object", "bool"]).columns

for col in numeric_cols:
    X_reg[col] = X_reg[col].fillna(X_reg[col].median())

for col in categorical_cols:
    X_reg[col] = X_reg[col].fillna(X_reg[col].mode()[0])

# One-hot encode categorical columns
X_reg = pd.get_dummies(X_reg, drop_first=True)

# Split data
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.30,
    random_state=42
)

# Grid search parameters
param_grid_reg = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2]
}

xgb_model = XGBRegressor(
    random_state=42,
    objective="reg:squarederror",
    reg_alpha=1,
    reg_lambda=1
)

grid_search_reg = GridSearchCV(
    xgb_model,
    param_grid_reg,
    cv=5,
    scoring="r2"
)

grid_search_reg.fit(X_train_reg, y_train_reg)

best_xgb = grid_search_reg.best_estimator_

y_pred_reg = best_xgb.predict(X_test_reg)

mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, y_pred_reg)

print("Best Parameters:")
print(grid_search_reg.best_params_)

print("\nMean Squared Error:", mse)
print("Root Mean Squared Error:", rmse)
print("R-squared Score:", r2)

## Regression Task: Optimized XGBoost Regressor

### Tuning with GridSearchCV and L1/L2 regularization.

C:\Users\Morgan Moore\AppData\Local\Temp\ipykernel_16220\3368761915.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_reg.select_dtypes(include=["object", "bool"]).columns


Best Parameters:
{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 50}

Mean Squared Error: 2761.8302356528507
Root Mean Squared Error: 52.55311822958606
R-squared Score: 0.031161748863931527


## Regression Results

The XGBoost model was optimized using GridSearchCV to identify the best combination of learning rate, maximum tree depth, and number of estimators. L1 and L2 regularization were also applied to reduce overfitting and improve model generalization.

### Performance Metrics

- Best Parameters:
  - Learning Rate: 0.01
  - Max Depth: 3
  - Number of Estimators: 50
- Mean Squared Error (MSE): 2761.83
- Root Mean Squared Error (RMSE): 52.55
- R-squared (R²): 0.0312

### Findings

The optimized model showed improvement compared to the original regression model. The Mean Squared Error and Root Mean Squared Error both decreased, indicating more accurate cholesterol predictions. The R-squared value improved from a negative value in the original model to a positive value after optimization, suggesting that the tuned model was better able to explain variation in cholesterol levels.

In [12]:
display(Markdown("## Clustering Task: Optimized K-Means Model"))
display(Markdown("### Tuning k values and applying PCA for dimensionality reduction."))

# Select clustering features
cluster_features = [
    "age",
    "trestbps",
    "chol",
    "thalch",
    "oldpeak",
    "ca"
]

X_cluster = heart[cluster_features].copy()

# Fill missing values
for col in X_cluster.columns:
    X_cluster[col] = X_cluster[col].fillna(X_cluster[col].median())

# Scale clustering data
cluster_scaler = StandardScaler()
X_cluster_scaled = cluster_scaler.fit_transform(X_cluster)

# Tune k values
silhouette_scores = {}

for k in range(2, 8):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    cluster_labels = kmeans.fit_predict(X_cluster_scaled)
    
    score = silhouette_score(
        X_cluster_scaled,
        cluster_labels
    )
    
    silhouette_scores[k] = score

print("Silhouette Scores Before PCA:")
for k, score in silhouette_scores.items():
    print(f"k={k}: {score:.4f}")

best_k = max(silhouette_scores, key=silhouette_scores.get)

print(f"\nBest k before PCA: {best_k}")
print(f"Best silhouette score before PCA: {silhouette_scores[best_k]:.4f}")

# Apply PCA as regularization/dimensionality reduction
pca = PCA(n_components=2)

X_cluster_pca = pca.fit_transform(X_cluster_scaled)

# Tune k values after PCA
pca_silhouette_scores = {}

for k in range(2, 8):
    kmeans_pca = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    pca_labels = kmeans_pca.fit_predict(X_cluster_pca)
    
    pca_score = silhouette_score(
        X_cluster_pca,
        pca_labels
    )
    
    pca_silhouette_scores[k] = pca_score

print("\nSilhouette Scores After PCA:")
for k, score in pca_silhouette_scores.items():
    print(f"k={k}: {score:.4f}")

best_k_pca = max(pca_silhouette_scores, key=pca_silhouette_scores.get)

print(f"\nBest k after PCA: {best_k_pca}")
print(f"Best silhouette score after PCA: {pca_silhouette_scores[best_k_pca]:.4f}")

## Clustering Task: Optimized K-Means Model

### Tuning k values and applying PCA for dimensionality reduction.

Silhouette Scores Before PCA:
k=2: 0.1851
k=3: 0.2123
k=4: 0.2241
k=5: 0.2042
k=6: 0.1983
k=7: 0.1864

Best k before PCA: 4
Best silhouette score before PCA: 0.2241

Silhouette Scores After PCA:
k=2: 0.3497
k=3: 0.3918
k=4: 0.3582
k=5: 0.3535
k=6: 0.3615
k=7: 0.3672

Best k after PCA: 3
Best silhouette score after PCA: 0.3918


## Clustering Results

The K-Means clustering model was optimized by testing multiple values of k and evaluating cluster quality using silhouette scores. Principal Component Analysis (PCA) was applied to reduce dimensionality and remove noise from the dataset.

### Performance Metrics

Before PCA:
- Best Number of Clusters (k): 4
- Silhouette Score: 0.2241

After PCA:
- Best Number of Clusters (k): 3
- Silhouette Score: 0.3918

### Findings

The silhouette score improved from 0.2241 to 0.3918 after PCA was applied, indicating better cluster separation and more meaningful patient groupings. The optimized clustering model produced stronger results than the original model, demonstrating that dimensionality reduction helped the algorithm focus on the most important cardiovascular risk factors while reducing noise in the dataset.